In [ ]:
from ollama import chat
from ollama import ChatResponse
import secrets
from Crypto.Cipher import AES
from sklearn.cluster import SpectralClustering
from sklearn.model_selection import train_test_split
from Crypto.Cipher import AES
from Crypto.Hash import SHAKE128
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPool2D, Attention, Normalization, Reshape, Conv2DTranspose
import polars as pl
import os
from Crypto.Util.Padding import pad, unpad
import math
from keras import Model, Input, Sequential
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'ollama'

In [3]:
'''Passwords & Hashes'''
passwords = [b"Andromeda", b"Neptune", b"Jupiter", b"Saturn"]
algs= ['128', '256'] #be sure to add 512 eventually
hash128 = {password: SHAKE128.new(password).read(128 // 8) for password in passwords}
hash256 = {password: SHAKE128.new(password).read(256 // 8) for password in passwords}
hash512 = {password: SHAKE128.new(password).read(512 // 8) for password in passwords}
print(hash512[b'Andromeda'])
print(hash128)

b"p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00%\xf0R\xf73NF\xb3\x86\xd2\x9dJ\x84\x9b(^\x8c\xc3a\x93\xf2\xca\xc6\xafI\xaf\x93\x1f\xe6\x8d\x05)\xb5\xa1\x08\x86X\x02\xc9'\x07\x84\xf3bl\\\x99f"
{b'Andromeda': b'p\x17Y{\xbc\x82\x06\xb7\tr\xd7\xeb)~\xf4\x00', b'Neptune': b'\xe3\xbc\xf4\x89\xd6\x13I4\xe9\x08&\xfd\xe2a[#', b'Jupiter': b'/Z\xd0\xd26\x95\x9b\xe5u\x9bs\xbcub-\x86', b'Saturn': b'L4 \xab\x95\n\xb2\xffM\x14O-\xf1\xe2a\x14'}


In [ ]:
'''Encrypt Messages'''
'''need all ciphertexts to be the same size, maybe just set a block limit based on the shortest message length?'''
#should only need to be done once
def encryptMessages(size, pass_dict, max_blocks):
    max_bytes = max_blocks * 512 // 8 #should be a perfect division anyways
    passwords = list(pass_dict.keys())
    for (root, dirs, files) in os.walk("../HumanReadable"):
        for file in files:
            with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
                plaintext = plaintextstore.read()
                for password in passwords:
                    #print(len(pass_dict[password]))
                    with open(f'../{size}ciphertext/{size}_{password}_{file}', 'xb') as cipherstore:
                        ciphertext = AES.new(pass_dict[password], AES.MODE_ECB).encrypt(pad(plaintext[:max_bytes], 512, 'iso7816'))
                        cipherstore.write(ciphertext)
#not most blocks present but most blocks to use = num blocks in shortest message
max_blocks = 1
image_shape = [8, 8]
for (root, dirs, files) in os.walk("../HumanReadable"):
    for file in files:
        with open(f'../HumanReadable/{file}', 'rb') as plaintextstore:
            plaintext = plaintextstore.read()
            if math.ceil((len(plaintext) * 8) / 512) < max_blocks:
                max_blocks = math.ceil((len(plaintext) * 8) / 512)
for iteration in range(2, max_blocks + 1):
    #basically go through every additional block after 1
    #alternates between square and 1:2 rectangle
    if iteration % 2 == 0:
        image_shape[1] *= 2
    else:
        image_shape[0] *= 2
assert(image_shape[0] * image_shape[1] == (512 * max_blocks) / 8)
image_shape.append(1)
image_shape = tuple(image_shape)
print(max_blocks, image_shape)
encryptMessages('128', hash128, max_blocks)
encryptMessages('256', hash256, max_blocks)

1 (8, 8, 1)


In [118]:
'''Compile Training Data'''
columns = ['alg', 'password', 'plaintext', 'source_file', 'ciphertext']
data = []
for alg in algs:
    for password in passwords:
        for (root, dirs, files) in os.walk("../HumanReadable"):
            for source_file in files:
                with open(f'../HumanReadable/{source_file}', 'rb') as plaintextstore:
                    plaintext = plaintextstore.read()
                    with open(f'../{alg}ciphertext/{alg}_{password}_{source_file}', 'rb') as cipherstore:
                        #If there is an error run encryption again
                        ciphertext = cipherstore.read()
                        #print(len(ciphertext))
                        #normalize and want (l,w,d) shape
                        ciphertextimage = [[[int(byte)/255] for byte in ciphertext[row*image_shape[0] : row*image_shape[0] + image_shape[1]]] for row in range(image_shape[0])]
                        ciphertextimagearray = np.asarray(ciphertextimage, np.float32)
                        data.append([alg, password, plaintext, source_file, ciphertextimagearray])

df = pl.LazyFrame(data, columns, orient='row')
df = df.collect()
df = df.sample(fraction=1, shuffle= True)
df = df.to_dummies(columns[:-1])
train_size = math.floor( .9 * len(df))
test_size = len(df) - train_size
train, test = df.head(train_size), df.tail(test_size)
train_data, test_data = train['ciphertext'], test['ciphertext']
train_targets, test_targets = train.select(pl.exclude('ciphertext')), test.select(pl.exclude('ciphertext'))

In [122]:
train_data.to_list()[0].shape

(8, 8, 1)

In [ ]:
'''Autoencoder'''
#is mean pooling worth investigating
#will need to play around with amount of parameters
#should i include more than just the ciphertext?
encoder_input = Input(shape=image_shape)
#4x4 byte matrix is AES operator size seems fitting
#blocks*4 is the amount of AES-128 blocks which seems like a decent starting metric
encoder1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')(encoder_input)
encoder2 = MaxPool2D(pool_size=(4,4), strides=4)(encoder1)
encoder3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')(encoder2)
encoder4 = MaxPool2D(pool_size=(2,2), strides=4)(encoder3)
encoder5 = Flatten()(encoder4)
embedded_layer = Dense(4, activation='relu')(encoder5)
decoder1 = Reshape((2,2,1))(embedded_layer)
decoder2 = Conv2DTranspose(max_blocks, (4,4), padding='same', activation='relu')(decoder1)
decoder3 = Conv2DTranspose(max_blocks * 4, (4,4), padding='same', activation='relu')(decoder2)
decoder4 = Conv2D(1, (4, 4), activation="sigmoid", padding="same")(decoder3)

autoencoder = Model(encoder_input, decoder4)
encoder = Model(encoder_input, embedded_layer)


In [157]:
autoencoder.summary()

Model: "functional_37"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_23 (InputLayer)     │ (None, 8, 8, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_81 (Conv2D)              │ (None, 8, 8, 4)        │            68 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_46 (MaxPooling2D) │ (None, 2, 2, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_82 (Conv2D)              │ (None, 2, 2, 1)        │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_47 (MaxPooling2D) │ (None, 1, 1, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_23 (Flatten)            │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 4)              │             8 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_19 (Reshape)            │ (None, 2, 2, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_83 (Conv2D)              │ (None, 2, 2, 1)        │             5 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_84 (Conv2D)              │ (None, 1, 1, 1)        │            10 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 156 (624.00 B)

 Trainable params: 156 (624.00 B)

 Non-trainable params: 0 (0.00 B)

In [125]:
'''keras example uses binary_crossentropy but they also had binary images, description says its for binary classification though'''
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.fit(np.array(train_data.to_list()), np.array(train_data.to_list()), epochs = 10)

Epoch 1/10


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 8, 8, 1), output.shape=(None, 2, 2, 4)

In [ ]:
'''Supervised Classification'''
activation = ''
if len(algs) == 2:
    activation = 'sigmoid'
else:
    activation = 'softmax'
encoder_input = Input(shape=image_shape)
algclass1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')
algclass2 = MaxPool2D(pool_size=(4,4), strides=4)
algclass3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')
algclass4 = MaxPool2D(pool_size=(4,4), strides=4)
algclass5 = Flatten()
alg_prediction_layer = Dense(len(algs), activation=activation)
alg_classifier = Sequential([encoder_input, algclass1, algclass2, algclass3, algclass4, algclass5, alg_prediction_layer])

In [ ]:
#use sparse categorical entropy for loss function i think
#may need to resize?

In [43]:
alg_classifier.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 8, 8, 4)        │            68 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 2, 2, 4)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 2, 2, 1)        │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 1, 1, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 139 (556.00 B)

 Trainable params: 139 (556.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
'''Unsupervised Clustering'''
#make it more adaptive but rn 3 algs, 5 i think password, and some amount of plaintext
sc_algs = SpectralClustering(len(algs))
sc_pass = SpectralClustering(len(passwords))
sc_source = SpectralClustering(df['source_file'].n_unique())

3

In [ ]:
'''Supervised Confusion'''
#prolly same structure as alg classifier but with an output dimension to represent the amount of passwords
'''need to do one for each password maybe'''
'''could also just do it raw and see which algs get password guessed the most'''
encoder_input = Input(shape=image_shape)
passclass1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')
passclass2 = MaxPool2D(pool_size=(4,4), strides=4)
passclass3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')
passclass4 = MaxPool2D(pool_size=(4,4), strides=4)
passclass5 = Flatten()
pass_prediction_layer = Dense(len(passwords), activation='softmax')
pass_classifier = Sequential([encoder_input, passclass1, passclass2, passclass3, 
                             passclass4, passclass5, pass_prediction_layer])

In [ ]:
'''Supervised Diffusion'''
#same as above but with respect to the amount of sources
encoder_input = Input(shape=image_shape)
sourceclass1 = Conv2D(max_blocks*4, (4,4), padding='same', activation='relu')
sourceclass2 = MaxPool2D(pool_size=(4,4), strides=4)
sourceclass3 = Conv2D(max_blocks, (4,4), padding='same', activation='relu')
sourceclass4 = MaxPool2D(pool_size=(4,4), strides=4)
sourceclass5 = Flatten()
source_prediction_layer = Dense(len(passwords), activation='softmax')
source_classifier = Sequential([encoder_input, sourceclass1, sourceclass2, sourceclass3, 
                             sourceclass4, sourceclass5, source_prediction_layer])